In [1]:
import pandas as pd

In [2]:
filepath = "nrc_data_csv/chinese_nrc_combined.csv"
df = pd.read_csv(filepath)
df.head(12)

,word,anger,disgust,fear,joy,sadness,surprise
0,ASP,0,0,1,0,0,0
1,A片,0,1,0,0,0,0
2,alb,0,0,0,0,0,0
3,bugaboo,1,0,1,0,1,0
4,cad,1,1,0,0,0,0
5,fop,0,0,0,0,0,0
6,tiff,1,0,0,0,0,0
7,wop,1,0,0,0,0,0
8,wot,0,0,0,0,0,0
9,“真的”,0,0,0,1,0,0


In [3]:
from datasets import load_dataset
import pandas as pd

target_emotions = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]
random_state = 42

# Load German evaluation data

comb_df = pd.concat(
    map(pd.read_csv, ["xed_chinese.csv", "brighter_chinese.csv", "xed_tw.csv"]),
    ignore_index=True
)

comb_df["text"] = comb_df["text"].astype(str)
comb_df["emotion"] = comb_df["emotion"].astype(str)

print("Original German rows:", len(comb_df))
print("\nOriginal German class distribution:")
print(comb_df["emotion"].value_counts())


# Balance German data
target_n = comb_df["emotion"].value_counts().min()

balanced_parts = []
for emotion, group in comb_df.groupby("emotion"):
    balanced_parts.append(
        group.sample(n=target_n, random_state=random_state)
    )

comb_df = (
    pd.concat(balanced_parts, ignore_index=True)
    .sample(frac=1, random_state=random_state)
    .reset_index(drop=True)
)

print("\nBalanced German rows:", len(comb_df))
print("\nBalanced German class distribution:")
print(comb_df["emotion"].value_counts())

/Users/arsenimuravjov/miniforge3/envs/xed-llm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Original German rows: 5426

Original German class distribution:
emotion
anger       2144
joy         1732
sadness      830
surprise     311
disgust      208
fear         201
Name: count, dtype: int64

Balanced German rows: 1206

Balanced German class distribution:
emotion
anger       201
disgust     201
joy         201
fear        201
sadness     201
surprise    201
Name: count, dtype: int64


In [4]:
import spacy
nlp = spacy.load("zh_core_web_sm")

print(df)

          word  anger  disgust  fear  joy  sadness  surprise
0          ASP      0        0     1    0        0         0
1           A片      0        1     0    0        0         0
2          alb      0        0     0    0        0         0
3      bugaboo      1        0     1    0        1         0
4          cad      1        1     0    0        0         0
...        ...    ...      ...   ...  ...      ...       ...
17130      龙卷风      0        0     1    0        0         0
17131       龙头      0        0     0    0        0         0
17132       龙骨      0        0     0    0        0         0
17133   （健康）状况      0        0     0    0        0         0
17134   （健康）狀況      0        0     0    0        0         0

[17135 rows x 7 columns]


In [5]:
import random

df["word"] = df["word"].astype(str).str.strip()

df = (
    df.groupby("word", as_index=False)[target_emotions]
    .max()
)

lexicon = df.set_index("word")[target_emotions].to_dict(orient="index")

emotion_frequency = {}

for word, scores in lexicon.items():
    active_count = sum(scores[emotion] for emotion in target_emotions)
    emotion_frequency[word] = 1 / active_count if active_count > 0 else 0

predictions = []

for text in comb_df["text"]:
    count = {emotion: 0 for emotion in target_emotions}

    doc = nlp(str(text))

    for token in doc:
        word = token.text.strip()

        if word in lexicon:
            word_weight = emotion_frequency[word]

            for emotion in target_emotions:
                count[emotion] += lexicon[word][emotion] * word_weight

    maxim = max(count.values())

    if maxim == 0:
        pred = "unknown"
    else:
        best_emotions = [emo for emo, score in count.items() if score == maxim]
        pred = random.choice(best_emotions)

    predictions.append(pred)

comb_df["prediction"] = predictions


In [6]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report, confusion_matrix
import pandas as pd
import numpy as np

target_emotions = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]

def evaluate_lexicon_predictions(df, gold_col="emotion", pred_col="prediction"):
    """
    Evaluates lexicon predictions.
    Unknown predictions are counted as wrong in the main evaluation.
    Also reports coverage and covered-only performance.
    """

    eval_df = df.copy()
    eval_df[gold_col] = eval_df[gold_col].astype(str)
    eval_df[pred_col] = eval_df[pred_col].astype(str)

    # -------------------------
    # Main evaluation: unknown = wrong
    # -------------------------
    y_true = eval_df[gold_col]
    y_pred = eval_df[pred_col]

    print("=== Main evaluation: unknown counted as wrong ===")
    print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
    print("Balanced accuracy:", round(balanced_accuracy_score(y_true, y_pred), 4))
    print("Macro F1:", round(f1_score(y_true, y_pred, labels=target_emotions, average="macro"), 4))
    print("Weighted F1:", round(f1_score(y_true, y_pred, labels=target_emotions, average="weighted"), 4))
    print()

    print(classification_report(
        y_true,
        y_pred,
        labels=target_emotions,
        digits=3,
        zero_division=0
    ))

    cm = confusion_matrix(y_true, y_pred, labels=target_emotions)
    cm_df = pd.DataFrame(cm, index=target_emotions, columns=target_emotions)

    print("Confusion matrix:")
    print(cm_df)
    print()

    # -------------------------
    # Coverage
    # -------------------------
    covered_mask = eval_df[pred_col] != "unknown"
    coverage = covered_mask.mean()

    print("=== Coverage ===")
    print("Coverage:", round(coverage, 4))
    print("Covered examples:", covered_mask.sum())
    print("Total examples:", len(eval_df))
    print()

    # -------------------------
    # Covered-only evaluation
    # -------------------------
    covered_df = eval_df[covered_mask].copy()

    if len(covered_df) > 0:
        print("=== Covered-only evaluation ===")
        print("Accuracy:", round(accuracy_score(covered_df[gold_col], covered_df[pred_col]), 4))
        print("Balanced accuracy:", round(balanced_accuracy_score(covered_df[gold_col], covered_df[pred_col]), 4))
        print("Macro F1:", round(f1_score(
            covered_df[gold_col],
            covered_df[pred_col],
            labels=target_emotions,
            average="macro",
            zero_division=0
        ), 4))
        print()

        print(classification_report(
            covered_df[gold_col],
            covered_df[pred_col],
            labels=target_emotions,
            digits=3,
            zero_division=0
        ))
    else:
        print("No covered examples.")

    results = {
        "accuracy_unknown_wrong": accuracy_score(y_true, y_pred),
        "balanced_accuracy_unknown_wrong": balanced_accuracy_score(y_true, y_pred),
        "macro_f1_unknown_wrong": f1_score(y_true, y_pred, labels=target_emotions, average="macro", zero_division=0),
        "weighted_f1_unknown_wrong": f1_score(y_true, y_pred, labels=target_emotions, average="weighted", zero_division=0),
        "coverage": coverage,
    }

    if len(covered_df) > 0:
        results.update({
            "accuracy_covered_only": accuracy_score(covered_df[gold_col], covered_df[pred_col]),
            "balanced_accuracy_covered_only": balanced_accuracy_score(covered_df[gold_col], covered_df[pred_col]),
            "macro_f1_covered_only": f1_score(
                covered_df[gold_col],
                covered_df[pred_col],
                labels=target_emotions,
                average="macro",
                zero_division=0
            ),
        })

    return results

In [7]:
results = evaluate_lexicon_predictions(
    comb_df,
    gold_col="emotion",
    pred_col="prediction"
)

=== Main evaluation: unknown counted as wrong ===
Accuracy: 0.1625
Balanced accuracy: 0.1625
Macro F1: 0.1829
Weighted F1: 0.1829

              precision    recall  f1-score   support

       anger      0.299     0.114     0.165       201
     disgust      0.220     0.055     0.088       201
        fear      0.248     0.199     0.221       201
         joy      0.323     0.413     0.362       201
     sadness      0.184     0.095     0.125       201
    surprise      0.213     0.100     0.136       201

   micro avg      0.264     0.163     0.201      1206
   macro avg      0.248     0.163     0.183      1206
weighted avg      0.248     0.163     0.183      1206

Confusion matrix:
          anger  disgust  fear  joy  sadness  surprise
anger        23       10    47   33       20        18
disgust      13       11    26   30       13        14
fear         13        8    40   21       20         9
joy          10        6     7   83       16        16
sadness       7       10    23   

/Users/arsenimuravjov/miniforge3/envs/xed-llm/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Users/arsenimuravjov/miniforge3/envs/xed-llm/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
